# Camada BRONZE

**Objetivo desta camada:** armazenar o dado exatamente como veio da fonte, sem transformar o conteúdo.

**Fonte:** ANP - Produção de Petróleo e Gás Natural por Poço (dados abertos governamentais).

## 1. Configuração

In [0]:
CATALOGO   = "workspace"
VOLUME     = "arquivos"
CAMINHO_ARQUIVO = f"/Volumes/{CATALOGO}/bronze/{VOLUME}/data_producao_poco.xlsx"
ABA_EXCEL = 0

## 2. Criação de catálogo, schemas e volume

In [0]:
# O código tenta criar o catálogo. No caso de falha, segue-se usando o catálogo padrão "workspace" que já existe.
try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
    print(f"Catálogo '{CATALOGO}' pronto.")
except Exception as e:
    print(f"Não foi possível criar o catálogo '{CATALOGO}'.")
    print(f"Vai ser utilizado o catálogo padrão.")

# Criação dos schemas e volume.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.gold")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.bronze.{VOLUME}")

print("Schemas (bronze/silver/gold) e volume prontos.")

Catálogo 'workspace' pronto.
Schemas (bronze/silver/gold) e volume prontos.


## 3. Leitura do Excel cru


In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql import functions as F
import pandas as pd

pdf = pd.read_excel(CAMINHO_ARQUIVO, sheet_name=ABA_EXCEL, dtype=str)

# Remover espaços extras dos nomes das colunas
pdf.columns = [str(c).strip() for c in pdf.columns]

# Converter o DataFrame do pandas para DataFrame do Spark
df_bronze = spark.createDataFrame(pdf)

# Checar as colunas que vieram do Excel
print("Colunas encontradas no Excel:")
for c in df_bronze.columns:
    print(" -", c)

df_bronze.printSchema()
display(df_bronze.limit(10))

Colunas encontradas no Excel:
 - Período
 - Campo
 - Operadora atual
 - Poço ANP
 - Bacia
 - Localização
 - Nome da Instalação
 - Ambiente
 - Óleo (bbl/d)
 - Gás Natural (mil m3/d)
 - Condensado (bbl/d)
 - Água (bbl/d)
root
 |-- Período: string (nullable = true)
 |-- Campo: string (nullable = true)
 |-- Operadora atual: string (nullable = true)
 |-- Poço ANP: string (nullable = true)
 |-- Bacia: string (nullable = true)
 |-- Localização: string (nullable = true)
 |-- Nome da Instalação: string (nullable = true)
 |-- Ambiente: string (nullable = true)
 |-- Óleo (bbl/d): string (nullable = true)
 |-- Gás Natural (mil m3/d): string (nullable = true)
 |-- Condensado (bbl/d): string (nullable = true)
 |-- Água (bbl/d): string (nullable = true)



Período,Campo,Operadora atual,Poço ANP,Bacia,Localização,Nome da Instalação,Ambiente,Óleo (bbl/d),Gás Natural (mil m3/d),Condensado (bbl/d),Água (bbl/d)
2021/01,ABALONE,Shell Brasil,7-ABA-2D-ESS,Campos,Mar,FPSO ESPIRITO SANTO,Produção Pós-sal Mar,0,0,0,0
2021/01,ACAJÁ-BURIZINHO,PetroRecôncavo,1-BZ-1-BA,Recôncavo,Terra,Ponto de Coleta de Acajá - Burizinho,Produção Terra,1.856508435483871,0.009584193548387096,0,0
2021/01,ACAJÁ-BURIZINHO,PetroRecôncavo,3-BZ-3-BA,Recôncavo,Terra,Ponto de Coleta de Acajá - Burizinho,Produção Terra,0,0,0,0
2021/01,ACAUÃ,Mandacaru Energia,3-BRSA-330-RN,Potiguar,Terra,Estação Coletora de Acauã (Acauã),Produção Terra,3.0359330272645164,0.0004967741935483871,0,2.359940769941936
2021/01,ACAUÃ,Mandacaru Energia,7-AC-2-RN,Potiguar,Terra,Estação Coletora de Acauã (Acauã),Produção Terra,6.071866054529033,0.001003548387096774,0,0.8087782623064517
2021/01,ÁGUA GRANDE,3R Bahia,7-AG-120-BA,Recôncavo,Terra,Estação Olinda,Produção Terra,60.68408688000001,1.9494354838709678,0,1895.4731997425804
2021/01,ÁGUA GRANDE,3R Bahia,7-AG-122-BA,Recôncavo,Terra,null,Produção Terra,0,0,0,0
2021/01,ÁGUA GRANDE,3R Bahia,7-AG-135-BA,Recôncavo,Terra,Estação Coletora de Rio Ventura,Produção Terra,41.756222516129036,0.4067096774193548,0,3137.325576052903
2021/01,ÁGUA GRANDE,3R Bahia,7-AG-145-BA,Recôncavo,Terra,null,Produção Terra,0,0,0,0
2021/01,ÁGUA GRANDE,3R Bahia,7-AG-148-BA,Recôncavo,Terra,Estação Olinda,Produção Terra,26.275376929354838,0.4284290322580645,0,4087.367898531935


## 4. Gravar a tabela Delta na camada Bronze

In [0]:
(df_bronze.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.columnMapping.mode", "name")
    .saveAsTable(f"{CATALOGO}.bronze.producao_poco"))

print("Tabela bronze.producao_poco gravada.")

Tabela bronze.producao_poco gravada.


## 5. Conferência

In [0]:
total = spark.table(f"{CATALOGO}.bronze.producao_poco").count()
print(f"Linhas na Bronze: {total:,}")
display(spark.sql(f"SELECT * FROM {CATALOGO}.bronze.producao_poco LIMIT 10"))

Linhas na Bronze: 607,046


Período,Campo,Operadora atual,Poço ANP,Bacia,Localização,Nome da Instalação,Ambiente,Óleo (bbl/d),Gás Natural (mil m3/d),Condensado (bbl/d),Água (bbl/d)
2022/08,CANTO DO AMARO,3R Potiguar,7-CAM-1401-RN,Potiguar,Terra,Estação Coletora Central de Benfica,Produção Terra,13.721119066451614,1.1789290322580643,0,118.44158603612902
2022/08,CANTO DO AMARO,3R Potiguar,7-CAM-1402D-RN,Potiguar,Terra,Estação Coletora Canto do Amaro E,Produção Terra,35.87200381258065,0.005703225806451613,0,0.21892596741935483
2022/08,CANTO DO AMARO,3R Potiguar,7-CAM-1404-RN,Potiguar,Terra,Estação Coletora Canto do Amaro E,Produção Terra,5.670770957741936,0.0009016129032258065,0,149.46211736774194
2022/08,CANTO DO AMARO,3R Potiguar,7-CAM-1405-RN,Potiguar,Terra,null,Produção Terra,0,0,0,0
2022/08,CANTO DO AMARO,3R Potiguar,7-CAM-1406D-RN,Potiguar,Terra,Estação Coletora Canto do Amaro H,Produção Terra,26.90618400322581,0.004277741935483871,0,2861.546016043548
2022/08,CANTO DO AMARO,3R Potiguar,7-CAM-1408-RN,Potiguar,Terra,null,Produção Terra,0,0,0,0
2022/08,CANTO DO AMARO,3R Potiguar,7-CAM-1409-RN,Potiguar,Terra,Estação Coletora Central de Benfica,Produção Terra,3.2966720283870967,0.06288677419354839,0,60.018178608387096
2022/08,CANTO DO AMARO,3R Potiguar,7-CAM-1410D-RN,Potiguar,Terra,null,Produção Terra,0,0,0,0
2022/08,CANTO DO AMARO,3R Potiguar,7-CAM-1411D-RN,Potiguar,Terra,Estação Coletora Alto da Pedra B,Produção Terra,12.846835476451613,0.0020425806451612903,0,135.39262398612905
2022/08,CANTO DO AMARO,3R Potiguar,7-CAM-1412-RN,Potiguar,Terra,Estação Coletora Alto da Pedra B,Produção Terra,16.639996700645163,0.002645483870967742,0,278.7862920864516
